In [ ]:
import pandas as pd
import numpy as np
import os
import sys
import joblib
import tsfel
from scipy.signal import find_peaks
import tensorflow as tf
from tensorflow.keras.models import load_model
import json
import warnings
import tkinter as tk
from tkinter import ttk, filedialog, messagebox, scrolledtext
import threading
from datetime import datetime
import traceback

warnings.filterwarnings('ignore')

# 获取资源路径（适配PyInstaller打包）
def get_resource_path(relative_path):
    """获取资源文件的绝对路径，适配PyInstaller打包后的环境"""
    try:
        # PyInstaller创建临时文件夹，将路径存储在_MEIPASS中
        base_path = sys._MEIPASS
    except Exception:
        base_path = os.path.abspath(".")
    return os.path.join(base_path, relative_path)

# ==================== 特征提取函数（与训练时完全一致）====================
def calculate_displacement_slope_and_detect_phases(time, displacement):
    """计算位移斜率并按规则划分阶段."""
    slope = np.gradient(displacement, time)
    abs_slope = np.abs(slope)

    phases = {}

    # 规则 1: 在10-30s内，第一个斜率数值超过3的点
    range_10_30 = (time >= 10) & (time <= 30)
    point1_index = np.argmax((slope > 3) & range_10_30)
    phases['Phase 1'] = time[point1_index] if slope[point1_index] > 3 else None

    # 规则 2: 在100s之前，最后一个斜率绝对值超过5的点
    range_before_100 = time < 100
    valid_indices_before_100 = np.where((abs_slope > 5) & range_before_100)[0]
    point2_index = valid_indices_before_100[-1] if len(valid_indices_before_100) > 0 else None
    phases['Phase 2'] = time[point2_index] if point2_index is not None else None

    # 规则 3: 在100s之后，第一个斜率绝对值超过40的点
    range_after_100 = time > 100
    valid_indices_after_100 = np.where((abs_slope > 40) & range_after_100)[0]
    point3_index = valid_indices_after_100[0] if len(valid_indices_after_100) > 0 else None
    phases['Phase 3'] = time[point3_index] if point3_index is not None else None

    return phases

def process_file_for_prediction(file_path):
    """为预测处理单个文件（与训练时的process_file函数一致，但不需要label）"""
    try:
        df = pd.read_csv(file_path)

        # 确保所有列都是数值类型
        df['TIME'] = pd.to_numeric(df['TIME'], errors='coerce')
        df['PRESSURE'] = pd.to_numeric(df['PRESSURE'], errors='coerce')
        df['CURRENT'] = pd.to_numeric(df['CURRENT'], errors='coerce')
        df['DISPLACEMENT'] = pd.to_numeric(df['DISPLACEMENT'], errors='coerce')

        # 删除包含NaN值的行
        df = df.dropna()

        if df.empty:
            return None

        # 使用位移的斜率划分阶段
        phases = calculate_displacement_slope_and_detect_phases(df['TIME'].values, df['DISPLACEMENT'].values)

        # 自动时间分割逻辑（结合阶段划分）
        segments = {
            'stage_1': df[(df['TIME'] <= phases['Phase 1'])] if phases['Phase 1'] else pd.DataFrame(),
            'stage_2': df[(df['TIME'] > phases['Phase 1']) & (df['TIME'] <= phases['Phase 2'])] if phases['Phase 2'] else pd.DataFrame(),
            'stage_3': df[(df['TIME'] > phases['Phase 2']) & (df['TIME'] <= phases['Phase 3'])] if phases['Phase 3'] else pd.DataFrame(),
            'stage_4': df[(df['TIME'] > phases['Phase 3'])] if phases['Phase 3'] else pd.DataFrame()
        }

        all_features = []

        for stage_name, segment in segments.items():
            if segment.empty:
                continue

            # 提取特征
            cfg = tsfel.get_features_by_domain()
            features = tsfel.time_series_features_extractor(cfg, segment[['PRESSURE', 'CURRENT', 'DISPLACEMENT']], verbose=0)

            # 为特征添加前缀以区分不同阶段的特征
            features.columns = [f"{stage_name}_{col}" for col in features.columns]
            all_features.append(features)

        if all_features:
            # 合并所有阶段的特征
            combined_features = pd.concat(all_features, axis=1)
            return combined_features
        else:
            return None

    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {e}")
        return None

# ==================== 预测器类 ====================
class WeldingPredictor:
    def __init__(self):
        """初始化预测器"""
        self.model = None
        self.scaler = None
        self.feature_names = None
        self.threshold = 0.5

    def load_model_and_components(self):
        """加载模型和相关组件"""
        try:
            # 1. 加载模型
            model_path = get_resource_path('model/nn_model-1119.keras')
            if os.path.exists(model_path):
                self.model = load_model(model_path)
                print(f"模型加载成功: {model_path}")
            else:
                raise FileNotFoundError(f"模型文件不存在: {model_path}")

            # 2. 加载标准化器
            scaler_path = get_resource_path('scaler-1119.joblib')
            if os.path.exists(scaler_path):
                self.scaler = joblib.load(scaler_path)
                print(f"标准化器加载成功: {scaler_path}")
            else:
                raise FileNotFoundError(f"标准化器文件不存在: {scaler_path}")

            # 3. 加载特征名称
            feature_names_path = get_resource_path('features/feature_names-1119.csv')
            if os.path.exists(feature_names_path):
                try:
                    feature_df = pd.read_csv(feature_names_path)
                    if len(feature_df.columns) == 1:
                        column_name = feature_df.columns[0]
                        if column_name == '0':
                            self.feature_names = feature_df.iloc[:, 0].tolist()
                        else:
                            self.feature_names = [column_name] + feature_df.iloc[:, 0].tolist()
                    else:
                        self.feature_names = feature_df.columns.tolist()
                    print(f"特征名称加载成功: {len(self.feature_names)} 个特征")
                except Exception as e:
                    print(f"特征名称加载失败: {e}")
                    feature_df = pd.read_csv(feature_names_path, header=None)
                    self.feature_names = feature_df.iloc[:, 0].tolist()
            else:
                print(f"特征名称文件不存在: {feature_names_path}")
                self.feature_names = None

            return True

        except Exception as e:
            print(f"加载模型组件时出错: {e}")
            return False

    def align_features(self, extracted_features):
        """特征对齐"""
        if self.feature_names is None:
            return extracted_features.fillna(0)

        # 按训练时的特征顺序构建对齐的特征
        aligned_features = pd.DataFrame(index=extracted_features.index)

        for feature_name in self.feature_names:
            if feature_name in extracted_features.columns:
                aligned_features[feature_name] = extracted_features[feature_name]
            else:
                aligned_features[feature_name] = 0

        return aligned_features.fillna(0)

    def predict_single_file(self, file_path):
        """预测单个文件"""
        try:
            # 1. 特征提取
            features = process_file_for_prediction(file_path)
            if features is None:
                return {
                    'filename': os.path.basename(file_path),
                    'prediction': None,
                    'probability': None,
                    'confidence': None,
                    'status': 'feature_extraction_failed'
                }

            # 2. 特征对齐
            aligned_features = self.align_features(features)

            # 3. 数据标准化
            features_scaled = self.scaler.transform(aligned_features.values)

            # 4. 模型预测
            prediction_prob = self.model.predict(features_scaled, verbose=0)[0][0]

            # 5. 二分类预测
            prediction = 1 if prediction_prob > self.threshold else 0

            # 6. 计算置信度
            confidence = max(prediction_prob, 1 - prediction_prob)

            # 7. 预测标签
            prediction_label = "好焊接" if prediction == 1 else "差焊接"

            return {
                'filename': os.path.basename(file_path),
                'prediction': prediction,
                'prediction_label': prediction_label,
                'probability': float(prediction_prob),
                'confidence': float(confidence),
                'status': 'success'
            }

        except Exception as e:
            return {
                'filename': os.path.basename(file_path),
                'prediction': None,
                'probability': None,
                'confidence': None,
                'status': f'prediction_failed: {str(e)}'
            }

    def predict_folder(self, folder_path, progress_callback=None):
        """预测文件夹中的所有CSV文件"""
        if not os.path.exists(folder_path):
            return []

        csv_files = [f for f in os.listdir(folder_path) if f.endswith('.csv')]
        csv_file_paths = [os.path.join(folder_path, f) for f in csv_files]

        if not csv_files:
            return []

        results = []
        for i, file_path in enumerate(csv_file_paths):
            result = self.predict_single_file(file_path)
            results.append(result)
            
            if progress_callback:
                progress_callback(i + 1, len(csv_files), os.path.basename(file_path))

        return results

# ==================== GUI界面 ====================
class WeldingPredictionGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("焊接质量预测系统 v1.0")
        self.root.geometry("800x600")
        
        # 设置主题
        style = ttk.Style()
        style.theme_use('clam')
        
        self.predictor = WeldingPredictor()
        self.setup_ui()
        
        # 自动加载模型
        self.load_model()

    def setup_ui(self):
        # 主框架
        main_frame = ttk.Frame(self.root, padding="10")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # 标题
        title_label = ttk.Label(main_frame, text="焊接质量预测系统", font=("Arial", 16, "bold"))
        title_label.grid(row=0, column=0, columnspan=3, pady=(0, 20))

        # 模型状态
        self.model_status_var = tk.StringVar(value="模型状态: 未加载")
        model_status_label = ttk.Label(main_frame, textvariable=self.model_status_var)
        model_status_label.grid(row=1, column=0, columnspan=3, pady=(0, 10))

        # 文件夹选择
        folder_frame = ttk.LabelFrame(main_frame, text="选择预测文件夹", padding="5")
        folder_frame.grid(row=2, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=(0, 10))
        
        self.folder_path_var = tk.StringVar()
        folder_entry = ttk.Entry(folder_frame, textvariable=self.folder_path_var, width=60)
        folder_entry.grid(row=0, column=0, padx=(0, 5))
        
        browse_button = ttk.Button(folder_frame, text="浏览", command=self.browse_folder)
        browse_button.grid(row=0, column=1)

        # 预测设置
        settings_frame = ttk.LabelFrame(main_frame, text="预测设置", padding="5")
        settings_frame.grid(row=3, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=(0, 10))
        
        ttk.Label(settings_frame, text="预测阈值:").grid(row=0, column=0, padx=(0, 5))
        self.threshold_var = tk.DoubleVar(value=0.5)
        threshold_scale = ttk.Scale(settings_frame, from_=0.1, to=0.9, orient=tk.HORIZONTAL, 
                                   variable=self.threshold_var, length=200)
        threshold_scale.grid(row=0, column=1, padx=(0, 5))
        
        self.threshold_label_var = tk.StringVar(value="0.5")
        threshold_label = ttk.Label(settings_frame, textvariable=self.threshold_label_var)
        threshold_label.grid(row=0, column=2)
        
        # 绑定阈值变化事件
        self.threshold_var.trace('w', self.update_threshold_label)

        # 预测按钮
        predict_button = ttk.Button(main_frame, text="开始预测", command=self.start_prediction)
        predict_button.grid(row=4, column=0, columnspan=3, pady=(0, 10))

        # 进度条
        self.progress_var = tk.DoubleVar()
        self.progress_bar = ttk.Progressbar(main_frame, variable=self.progress_var, maximum=100)
        self.progress_bar.grid(row=5, column=0, columnspan=3, sticky=(tk.W, tk.E), pady=(0, 5))

        # 状态标签
        self.status_var = tk.StringVar(value="就绪")
        status_label = ttk.Label(main_frame, textvariable=self.status_var)
        status_label.grid(row=6, column=0, columnspan=3, pady=(0, 10))

        # 结果显示区域
        result_frame = ttk.LabelFrame(main_frame, text="预测结果", padding="5")
        result_frame.grid(row=7, column=0, columnspan=3, sticky=(tk.W, tk.E, tk.N, tk.S), pady=(0, 10))
        
        self.result_text = scrolledtext.ScrolledText(result_frame, height=15, width=80)
        self.result_text.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))

        # 保存结果按钮
        save_button = ttk.Button(main_frame, text="保存结果", command=self.save_results)
        save_button.grid(row=8, column=0, columnspan=3, pady=(10, 0))

        # 配置网格权重
        main_frame.columnconfigure(0, weight=1)
        main_frame.rowconfigure(7, weight=1)
        result_frame.columnconfigure(0, weight=1)
        result_frame.rowconfigure(0, weight=1)
        self.root.columnconfigure(0, weight=1)
        self.root.rowconfigure(0, weight=1)

        self.results = []

    def load_model(self):
        """加载模型"""
        try:
            if self.predictor.load_model_and_components():
                self.model_status_var.set("模型状态: 已加载")
                self.append_result("✅ 模型组件加载成功！")
            else:
                self.model_status_var.set("模型状态: 加载失败")
                self.append_result("❌ 模型组件加载失败！请检查模型文件是否存在。")
        except Exception as e:
            self.model_status_var.set("模型状态: 加载失败")
            self.append_result(f"❌ 加载模型时出错: {str(e)}")

    def browse_folder(self):
        """浏览文件夹"""
        folder_path = filedialog.askdirectory()
        if folder_path:
            self.folder_path_var.set(folder_path)

    def update_threshold_label(self, *args):
        """更新阈值标签"""
        value = self.threshold_var.get()
        self.threshold_label_var.set(f"{value:.2f}")

    def start_prediction(self):
        """开始预测"""
        folder_path = self.folder_path_var.get()
        if not folder_path:
            messagebox.showerror("错误", "请选择预测文件夹！")
            return

        if not os.path.exists(folder_path):
            messagebox.showerror("错误", "选择的文件夹不存在！")
            return

        if self.predictor.model is None:
            messagebox.showerror("错误", "模型未加载，无法进行预测！")
            return

        # 更新阈值
        self.predictor.threshold = self.threshold_var.get()

        # 在新线程中执行预测
        threading.Thread(target=self.run_prediction, args=(folder_path,), daemon=True).start()

    def run_prediction(self, folder_path):
        """运行预测（在后台线程中）"""
        try:
            self.status_var.set("正在预测...")
            self.progress_var.set(0)
            self.result_text.delete(1.0, tk.END)

            def progress_callback(current, total, filename):
                progress = (current / total) * 100
                self.progress_var.set(progress)
                self.status_var.set(f"正在预测: {filename} ({current}/{total})")

            # 执行预测
            self.results = self.predictor.predict_folder(folder_path, progress_callback)

            # 显示结果
            self.display_results()

            self.status_var.set("预测完成！")
            self.progress_var.set(100)

        except Exception as e:
            self.status_var.set("预测失败！")
            self.append_result(f"❌ 预测过程中出错: {str(e)}\n{traceback.format_exc()}")

    def display_results(self):
        """显示预测结果"""
        if not self.results:
            self.append_result("❌ 没有获得预测结果")
            return

        # 统计信息
        total_files = len(self.results)
        successful = [r for r in self.results if r['status'] == 'success']
        failed = [r for r in self.results if r['status'] != 'success']
        positive_predictions = [r for r in successful if r['prediction'] == 1]
        negative_predictions = [r for r in successful if r['prediction'] == 0]

        # 显示统计信息
        self.append_result("="*60)
        self.append_result("📊 预测结果统计:")
        self.append_result(f"  • 总文件数: {total_files}")
        self.append_result(f"  • 成功预测: {len(successful)}")
        self.append_result(f"  • 预测失败: {len(failed)}")
        if len(successful) > 0:
            self.append_result(f"  • 好焊接: {len(positive_predictions)} ({len(positive_predictions)/len(successful)*100:.1f}%)")
            self.append_result(f"  • 差焊接: {len(negative_predictions)} ({len(negative_predictions)/len(successful)*100:.1f}%)")

        self.append_result("\n📋 详细预测结果:")
        self.append_result("-" * 60)
        self.append_result(f"{'文件名':<30} {'预测结果':<10} {'概率':<8} {'置信度':<8}")
        self.append_result("-" * 60)

        for result in self.results:
            if result['status'] == 'success':
                filename = result['filename'][:28] + ".." if len(result['filename']) > 30 else result['filename']
                pred_label = result['prediction_label']
                prob = f"{result['probability']:.3f}"
                conf = f"{result['confidence']:.1%}"
                
                self.append_result(f"{filename:<30} {pred_label:<10} {prob:<8} {conf:<8}")
            else:
                filename = result['filename'][:28] + ".." if len(result['filename']) > 30 else result['filename']
                self.append_result(f"{filename:<30} {'失败':<10} {'N/A':<8} {'N/A':<8}")

        self.append_result("="*60)

    def append_result(self, text):
        """追加结果文本"""
        self.result_text.insert(tk.END, text + "\n")
        self.result_text.see(tk.END)

    def save_results(self):
        """保存预测结果"""
        if not self.results:
            messagebox.showwarning("警告", "没有预测结果可保存！")
            return

        # 选择保存位置
        file_path = filedialog.asksaveasfilename(
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )

        if file_path:
            try:
                # 保存为CSV
                results_df = pd.DataFrame(self.results)
                results_df.to_csv(file_path, index=False, encoding='utf-8')
                
                # 同时保存详细报告
                report_path = file_path.replace('.csv', '_report.txt')
                with open(report_path, 'w', encoding='utf-8') as f:
                    f.write(self.result_text.get(1.0, tk.END))
                
                messagebox.showinfo("成功", f"结果已保存到:\n{file_path}\n{report_path}")
            except Exception as e:
                messagebox.showerror("错误", f"保存失败: {str(e)}")

def main():
    """主函数"""
    try:
        root = tk.Tk()
        app = WeldingPredictionGUI(root)
        root.mainloop()
    except Exception as e:
        print(f"程序启动失败: {e}")
        traceback.print_exc()

if __name__ == "__main__":
    main()

加载模型组件时出错: 模型文件不存在: /Users/dawn/Desktop/成都材料院/钢轨焊接工艺参数分析/model/nn_model-1119.keras


2025-09-27 10:21:52.153 python[73153:2038729] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


: 

In [ ]:
# 加载模型组件时出错: 模型文件不存在: /Users/dawn/Desktop/成都材料院/钢轨焊接工艺参数分析/model/nn_model-1119.keras
# 2025-09-27 10:21:52.153 python[73153:2038729] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'